In [1]:
import os
import re
import fitz
from pathlib import Path
from typing import List
import pandas as pd
import pymupdf4llm

from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
from langchain_ollama import OllamaEmbeddings, OllamaLLM
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_ollama.llms import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate

In [2]:
DATA_DIR = '../school_knowledge_base'
CHROMA_DIR = '../chroma_db'

In [3]:
def clean_text(text):
    """Функция для очистки текста"""
    # Убираем лишние переносы строк внутри абзацев
    text = re.sub(r'(?<=[^\.!?])\n+(?=[а-яёa-z])', ' ', text)
    # убираем множественные пробелы
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [4]:
def clean_school_md(text):
    # 1. Удаляем блоки с описанием пропущенных картинок
    text = re.sub(r'==> picture \[.*?\] intentionally omitted <==', '', text)
    text = re.sub(r'----- Start of picture text -----.*?----- End of picture text -----', '', text, flags=re.DOTALL)
    
    # 2. Удаляем техническую информацию о цифровой подписи (часто в начале)
    # Ищем блок от "РАССМОТРЕНО" до названия документа
    text = re.sub(r'РАССМОТРЕНО.*?Дата: \d{4}\.\d{2}\.\d{2}.*?\+03\'00\'', '', text, flags=re.DOTALL)
    
    # 3. Удаляем номера страниц (например, _ 2 _, _ 3 _)
    text = re.sub(r'_\s\d+\s_', '', text)
    
    # 4. Убираем множественные пустые строки (оставляем максимум две для структуры)
    text = re.sub(r'\n{3,}', '\n\n', text)
    
    # 5. Убираем лишние пробелы в начале и конце строк
    text = "\n".join([line.strip() for line in text.split('\n')])
    
    return text.strip()

In [5]:
data_md = []
data_pdf = []
    
for root, _, files in os.walk(DATA_DIR):
    for file in files:
        path = os.path.join(root, file)
        extension = file.split('.')[-1].lower()
        text = ''

        try:
            if extension == 'md':
                with open(path, 'r', encoding='utf-8') as f:
                    text = f.read()
                    if text:
                        data_md.append(Document(
                            page_content=text,
                            metadata={
                                "source": file,
                                "path": path,
                                "format": extension,
                                "size_bytes": os.path.getsize(path)
                            }
                        ))

            elif extension == 'pdf':
                text = pymupdf4llm.to_markdown(path)
                text = clean_school_md(text)
                if text:
                    data_pdf.append(Document(
                        page_content=text,
                        metadata={
                            "source": file,
                            "path": path,
                            "format": extension,
                            "size_bytes": os.path.getsize(path)
                        }
                    ))

        except Exception as e:
            print(f'Ошибка при чтении файла {file}: {e}')

print(f'Всего загружено документов: {len(data_md) + len(data_pdf)}')

Всего загружено документов: 67


In [6]:
# сплиттер для pdf
pdf_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=300,
    separators=["\n\n", "\n", ". ", "? ", "! ", " ", ""],
    length_function=len,
    add_start_index=True
)

# сплиттер для md
md_splitter = MarkdownHeaderTextSplitter(
    [("#", "Header 1")]
)

In [7]:
chunks = []

for doc in data_md:
    md_chunks = md_splitter.split_text(doc.page_content)
        
    for chunk in md_chunks:
        chunk.metadata.update(doc.metadata)

    # подстраховка по длине для больших секций
    chunks.extend(pdf_splitter.split_documents(md_chunks))

chunks.extend(pdf_splitter.split_documents(data_pdf))

print(f'Всего чанков: {len(chunks)}')

Всего чанков: 1768


In [8]:
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-small",
    model_kwargs={'device': 'cpu'}
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
print('Создание ChromaDB...')
db = Chroma(
    collection_name='school_knowledge_base',
    persist_directory=CHROMA_DIR,
    embedding_function=embeddings
)

batch_size = 10
for i in tqdm(range(0, len(chunks), batch_size), desc="Загрузка чанков в базу"):
    batch = chunks[i : i + batch_size]
    try:
        db.add_documents(batch)
    except Exception as e:
        print(f"\nОшибка на батче {i//batch_size}: {e}")
        if batch:
            print(f"Текст в проблемном батче: {batch[0].page_content[:100]}...")

print(f'ChromaDB создана в {CHROMA_DIR}')

Создание ChromaDB...


Загрузка чанков в базу: 100%|████████████████████████████████████████████████████████████████████████████| 177/177 [04:02<00:00,  1.37s/it]

ChromaDB создана в ../chroma_db


In [32]:
retriever = db.as_retriever(
    search_type='similarity', 
    search_kwargs={"k": 8}
)

In [33]:
questions = [
    'Во сколько начинается первый урок?',
    'Когда весенние каникулы?',
    'Зачем нужна внеурочная деятельность?',
    'Как я могу связаться с директором?',
    'Что на завтрак в 5 классе?',
    'Какие правила поведения в школе для родителей?'
]

models = ['qwen2.5:7b', 'llama3.2']

In [34]:
results = []
    
template = """Вы — экспертный аналитик базы знаний школы. 
Ваша цель: найти ответ на вопрос в предоставленных фрагментах документов.

КОНТЕКСТ:
{context}

ВОПРОС: {question}

ИНСТРУКЦИЯ:
1. Проанализируй контекст. Если информация представлена в виде списка, таблицы или расписания — изучи каждую строку.
2. Если в тексте упоминаются похожие термины (например, "питание" вместо "завтрак"), используй их для ответа.
3. Если ответ найден частично, напиши то, что удалось найти.
4. Сначала кратко опиши, что ты нашел в документах, а затем дай итоговый ответ.

ОТВЕТ:"""

prompt = ChatPromptTemplate.from_template(template)

for model in models:
    print(f"\n[{model}] Инициализация...")
    try:
        llm = OllamaLLM(model=model, temperature=0.1)
        chain = prompt | llm
    except Exception as e:
        print(f'[{model}] Ошибка при инициализации: {e}')
        continue 
        
    for q in questions:
        retrieved_docs = retriever.invoke(q)
        context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])
            
        try:
            answer = chain.invoke({'context': context_text, 'question': q})
        except Exception as e:
            print(f'Ошибка генерации ответа: {e}')
            answer = "Ошибка генерации"

        results.append({
            "model": model,
            "question": q,
            "context": context_text[:100] + "...", # cохраняем начало контекста для отладки
            "answer": answer.strip()
        })
        
    print(f"[{model}] Тестирование завершено!")


[qwen2.5:7b] Инициализация...
[qwen2.5:7b] Тестирование завершено!

[llama3.2] Инициализация...
[llama3.2] Тестирование завершено!


In [35]:
results_df = pd.DataFrame(results)

In [36]:
results_df

,model,question,context,answer
0,qwen2.5:7b,Во сколько начинается первый урок?,"**Учебный план –** документ, который определяе...","В документах указано, что начало уроков первой..."
1,qwen2.5:7b,Когда весенние каникулы?,Осенние - 25.10.2025 - 04.11.2025 Зимние - 31....,"В документах указано, что весенние каникулы дл..."
2,qwen2.5:7b,Зачем нужна внеурочная деятельность?,Внеурочная деятельность является составной час...,"В документах указано, что внеурочная деятельно..."
3,qwen2.5:7b,Как я могу связаться с директором?,"**2.3.** Права и обязанности обучающегося, пре...",В документах не содержится прямого ответа на в...
4,qwen2.5:7b,Что на завтрак в 5 классе?,## **Недельная сетка часов 1 е классы**\n\n|**...,В предоставленных документах нет информации о ...
5,qwen2.5:7b,Какие правила поведения в школе для родителей?,- бесцельного нахождения на прилегающей к школ...,В документах содержатся правила поведения для ...
6,llama3.2,Во сколько начинается первый урок?,"**Учебный план –** документ, который определяе...",В 1 классе начинается первый урок в 08:00.
7,llama3.2,Когда весенние каникулы?,Осенние - 25.10.2025 - 04.11.2025 Зимние - 31....,Весенние каникулы начались 28.03.2026 - 05.04....
8,llama3.2,Зачем нужна внеурочная деятельность?,Внеурочная деятельность является составной час...,Внеурочная деятельность необходима для развити...
9,llama3.2,Как я могу связаться с директором?,"**2.3.** Права и обязанности обучающегося, пре...",Из предоставленного текста можно сделать вывод...


In [40]:
results_df.iloc[11]['answer']

'Родителям (законным представителям) в школу разрешен вход по предварительной договоренности с администрацией или педагогами школы и оформлении разрешения на вход служебной запиской.'